# How to Use Different Data Sources

Dieses Notebook zeigt, wie man **einfach** zwischen verschiedenen Datenquellen wechselt.

## 3 Datenquellen verfügbar:

1. **Demo** - Kleine Beispieldaten (kein Setup nötig)
2. **BigQuery** - Google Cloud (vollständig, benötigt Credentials)
3. **Local** - Blockchair Dumps auf SSD (benötigt Download)

---

## Setup

In [ ]:
# Imports
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

# Import data config
from data_config import DataConfig, load_data

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, sum as spark_sum

print("✓ Imports erfolgreich")

In [ ]:
# Spark Session
spark = SparkSession.builder \
    .appName("Bitcoin Data Analysis") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"✓ Spark {spark.version} gestartet")

---

## 🎯 WÄHLE DATENQUELLE

**Einfach eine der folgenden Optionen auskommentieren:**

In [ ]:
# ============================================================================
# OPTION 1: DEMO-DATEN (schnell, kein Setup)
# ============================================================================

# config = DataConfig(source="demo")


# ============================================================================
# OPTION 2: BIGQUERY (Cloud, vollständig)
# ============================================================================

# config = DataConfig(source="bigquery")


# ============================================================================
# OPTION 3: LOKALE BLOCKCHAIR-DATEN (empfohlen)
# ============================================================================

config = DataConfig(
    source="local",
    local_data_path="/Volumes/MySSD/bitcoin_data/extracted"  # ANPASSEN!
)

# Tipp: Wenn du die Daten mit download_bitcoin_data.py heruntergeladen hast,
#       wird der Pfad automatisch erkannt und du kannst einfach schreiben:
# config = DataConfig(source="local")


# ============================================================================
# FERTIG! Ab hier ist alles gleich, egal welche Quelle.
# ============================================================================

print(f"\n{'='*70}")
print(f"Gewählte Datenquelle: {config}")
print(f"{'='*70}\n")

---

## 📊 Daten laden

**Der Code ist identisch, egal welche Datenquelle!**

In [ ]:
# Zeitraum definieren
START_DATE = "2021-01-01"
END_DATE = "2021-01-07"  # Eine Woche

# Daten laden (funktioniert mit allen 3 Quellen!)
df_transactions = load_data(
    config=config,
    start_date=START_DATE,
    end_date=END_DATE,
    spark=spark,
    filter_coinbase=True
)

print(f"\n✓ Daten geladen: {df_transactions.count():,} Transaktionen")

---

## 🔍 Analyse (gleich für alle Quellen)

In [ ]:
# Sample anzeigen
print("Sample Transactions:\n")
df_transactions.select("hash", "input_count", "output_count", "fee").show(10, truncate=False)

In [ ]:
# Multi-Input-Analyse
single_input = df_transactions.filter(col("input_count") == 1).count()
multi_input = df_transactions.filter(col("input_count") > 1).count()
total = df_transactions.count()

print(f"\n{'='*60}")
print(f"MULTI-INPUT ANALYSE")
print(f"{'='*60}")
print(f"Single-Input (1 Adresse):  {single_input:,} ({single_input/total*100:.1f}%)")
print(f"Multi-Input (≥2 Adressen): {multi_input:,} ({multi_input/total*100:.1f}%)")
print(f"Total:                     {total:,}")
print(f"{'='*60}")
print(f"\n💡 {multi_input/total*100:.1f}% nutzbar für Entity-Clustering!")

In [ ]:
# Statistiken
stats = df_transactions.select(
    avg("input_count").alias("avg_inputs"),
    avg("output_count").alias("avg_outputs"),
    avg("fee").alias("avg_fee")
).collect()[0]

print(f"\nStatistiken:")
print(f"  Durchschnittliche Inputs:  {stats['avg_inputs']:.2f}")
print(f"  Durchschnittliche Outputs: {stats['avg_outputs']:.2f}")
print(f"  Durchschnittliche Fee:     {stats['avg_fee']/100000000:.8f} BTC")

---

## 🔧 Erweiterte Nutzung (nur für lokale Daten)

Wenn du lokale Blockchair-Daten verwendest, hast du zusätzliche Optionen:

In [ ]:
# Nur wenn source="local"!
if config.source == "local":
    from data_config import get_loader

    # Hole Loader-Objekt
    loader = get_loader(config, spark=spark)

    # Direkt Multi-Input-Transaktionen laden (2-50 Inputs)
    df_multi = loader.get_multi_input_transactions(
        START_DATE, END_DATE,
        min_inputs=2,
        max_inputs=50
    )

    print(f"\n✓ Multi-Input Transactions: {df_multi.count():,}")

    # Lade auch Blocks und Outputs
    df_blocks = loader.load_blocks(START_DATE, END_DATE)
    df_outputs = loader.load_outputs(START_DATE, END_DATE, unspent_only=True)

    print(f"✓ Blocks: {df_blocks.count():,}")
    print(f"✓ UTXOs (unspent): {df_outputs.count():,}")

    # SQL Views erstellen
    loader.create_temp_views(START_DATE, END_DATE)

    print("\n✓ SQL Views erstellt: blocks, transactions, outputs")
    print("   Jetzt kannst du SQL-Queries ausführen!")

else:
    print("\n⚠️  Erweiterte Features nur für lokale Daten verfügbar")
    print("   Wechsle zu: config = DataConfig(source='local')")

In [ ]:
# SQL Query Beispiel (nur wenn SQL Views erstellt wurden)
if config.source == "local":
    result = spark.sql("""
        SELECT
            input_count,
            COUNT(*) as tx_count,
            AVG(fee) / 100000000 as avg_fee_btc
        FROM transactions
        WHERE input_count BETWEEN 1 AND 10
        GROUP BY input_count
        ORDER BY input_count
    """)

    print("\nSQL Query Ergebnis:\n")
    result.show()

---

## 📖 Zusammenfassung

### So wechselst du zwischen Datenquellen:

```python
# Demo
config = DataConfig(source="demo")

# BigQuery
config = DataConfig(source="bigquery")

# Lokal (automatische Pfad-Erkennung)
config = DataConfig(source="local")

# Lokal (manueller Pfad)
config = DataConfig(source="local", local_data_path="/pfad/zu/extracted")

# Dann immer gleich:
df = load_data(config, "2021-01-01", "2021-01-07", spark=spark)
```

### Vorteile:

- ✅ **Ein Code** für alle Datenquellen
- ✅ **Einfaches Switching** - nur 1 Zeile ändern
- ✅ **Automatische Pfad-Erkennung** für lokale Daten
- ✅ **Erweiterte Features** für lokale Daten (SQL Views, etc.)

### Nächste Schritte:

1. **Daten herunterladen** (falls noch nicht geschehen):
   ```bash
   python download_bitcoin_data.py
   ```

2. **In deinen Notebooks verwenden:**
   - Kopiere die Config-Zelle in dein Notebook
   - Wähle deine Datenquelle
   - Fertig!

3. **Entity-Clustering** (Notebook 02), **Whale-Detection** (Notebook 03), etc.